# 10_agent_with_memory

10_agent_with_memory.py — load_memory 도구로 과거 기억 능동 검색

에이전트가 *필요할 때* load_memory 를 호출해 과거 세션의 기억을 가져온다.
시나리오: 세션1 에서 음식 선호 적재 → 세션2 에서 load_memory 로 회상.

⚠ load_memory 의 검색 결과 포맷은 SDK 버전 의존. 본 데모는 *호출 형태만* 시연.

In [1]:
import os, sys, ssl, certifi
# Windows 인증서 저장소 손상 우회(임베딩/HTTPS 로드 SSL 에러 방지)
ssl.SSLContext.load_default_certs = lambda self, *a, **k: self.load_verify_locations(certifi.where())
# 노트북 커널엔 이미 이벤트 루프가 돌아 스크립트의 asyncio.run() 이 깨짐 → nest_asyncio 로 중첩 허용
import nest_asyncio; nest_asyncio.apply()
# 노트북 커널엔 __file__ 이 없으므로 스크립트 호환 위해 정의 + supp/ 를 import 경로에 추가
__file__ = os.path.join(os.getcwd(), '10_agent_with_memory.py')
sys.path.insert(0, os.path.abspath('..'))

In [2]:
"""
10_agent_with_memory.py — load_memory 도구로 과거 기억 능동 검색

에이전트가 *필요할 때* load_memory 를 호출해 과거 세션의 기억을 가져온다.
시나리오: 세션1 에서 음식 선호 적재 → 세션2 에서 load_memory 로 회상.

⚠ load_memory 의 검색 결과 포맷은 SDK 버전 의존. 본 데모는 *호출 형태만* 시연.
"""
import asyncio

from google.adk.agents import LlmAgent
from google.adk.memory import InMemoryMemoryService
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import load_memory
from google.genai.types import Content, Part

from _adk_common import adk_model, banner, adk_unavailable


async def demo() -> None:
    sess_svc = InMemorySessionService()
    mem_svc = InMemoryMemoryService()

    # ─── 세션 1: 사용자가 음식 선호 알려줌 ───
    s1 = await sess_svc.create_session(app_name="food_app", user_id="alice", session_id="d1")
    s1.state["favorite_food"] = "매운 떡볶이"
    await mem_svc.add_session_to_memory(s1)
    print(f"  📦 세션 1 완료 → 메모리 적재 (favorite_food='매운 떡볶이')")

    # ─── 세션 2: 새 에이전트가 load_memory 로 회상해 답변 ───
    agent = LlmAgent(
        name="memory_agent",
        model=adk_model(),
        instruction=(
            "사용자 선호를 알고 답변해야 한다. 모르면 load_memory 도구로 "
            "과거 대화를 검색해 참고하라. 한국어 한 문장."
        ),
        tools=[load_memory],
    )

    runner = Runner(
        agent=agent, app_name="food_app",
        session_service=sess_svc, memory_service=mem_svc,
    )
    await sess_svc.create_session(app_name="food_app", user_id="alice", session_id="d2")
    msg = Content(role="user", parts=[Part(text="내가 좋아하는 음식 스타일은?")])
    print(f"\n  ❓ d2 user : 내가 좋아하는 음식 스타일은?")
    final = ""
    async for event in runner.run_async(user_id="alice", session_id="d2", new_message=msg):
        if event.is_final_response() and event.content and event.content.parts:
            final = event.content.parts[0].text or ""
            break
    print(f"  💬 d2 bot  : {final[:200]}")


def main() -> None:
    banner("load_memory — 과거 세션 회상")
    try:
        asyncio.run(demo())
    except Exception as e:
        print(f"\n  ⚠ {type(e).__name__}: {str(e)[:200]}")
        adk_unavailable()


if __name__ == "__main__":
    main()


📌 load_memory — 과거 세션 회상
  📦 세션 1 완료 → 메모리 적재 (favorite_food='매운 떡볶이')

  ❓ d2 user : 내가 좋아하는 음식 스타일은?


D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\google\adk\models\llm_request.py:256: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()



Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



Node execution failed with exception
Traceback (most recent call last):
  File "D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\litellm\llms\custom_httpx\llm_http_handler.py", line 180, in _make_common_async_call
    response = await async_httpx_client.post(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\litellm\litellm_core_utils\logging_utils.py", line 297, in async_wrapper
    result = await func(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\litellm\llms\custom_httpx\http_handler.py", line 574, in post
    await _raise_masked_async_error(e, stream)
  File "D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\litellm\llms\custom_httpx\http_handler.py", line 363, in _raise_masked_async_error
    raise MaskedHTTPStatusError(e, message=_text, text=_text) from None
litellm.llms.custom_httpx.http_handler.MaskedHTTPStatusError: Cl

Root node memory_agent failed.
Traceback (most recent call last):
  File "D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\litellm\llms\custom_httpx\llm_http_handler.py", line 180, in _make_common_async_call
    response = await async_httpx_client.post(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\litellm\litellm_core_utils\logging_utils.py", line 297, in async_wrapper
    result = await func(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\litellm\llms\custom_httpx\http_handler.py", line 574, in post
    await _raise_masked_async_error(e, stream)
  File "D:\git\2604_agent_210h_handson\supp\venv\Lib\site-packages\litellm\llms\custom_httpx\http_handler.py", line 363, in _raise_masked_async_error
    raise MaskedHTTPStatusError(e, message=_text, text=_text) from None
litellm.llms.custom_httpx.http_handler.MaskedHTTPStatusError: Client e


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


  ⚠ RateLimitError: litellm.RateLimitError: RateLimitError: OpenrouterException - {"error":{"message":"Provider returned error","code":429,"metadata":{"raw":"google/gemma-4-26b-a4b-it:free is temporarily rate-limited ups
⚠️ ADK 실행 실패 — google-adk[extensions] 설치 + OPENROUTER_API_KEY 확인.
